In [10]:
"""
TIER 3 ONLY: LLM adjudication for whatever tier 1 and tier 2 couldn't
resolve automatically.

What this does, in plain terms:
  This is for the leftover ambiguous cases, entities that are similar
  in wording but where you can't automatically tell (via a code match)
  whether they're the same real-world thing or genuinely different.
  This includes:
    - the "skipped_state_conflict" rows from tier 2 (same code, but a
      modifier like "gel" differs, e.g. isolate vs its gelled form)
    - the "skipped_low_similarity" rows from tier 2
    - any entity with no extractable code at all, run through
      step1_generate_entity_candidates.py's similarity grouping first

  For each group, this script pulls the actual sentence(s) each entity
  appeared in and gives that context to the LLM, the same evidence you
  used manually to work out that "PPI" and "commercial pea isolate"
  were the same sample in one paper. The LLM is explicitly told it is
  allowed to say "these are NOT the same" rather than being pushed to
  force a merge.

  This is the smallest, most expensive-per-item tier, meant for the
  genuinely hard cases only, everything tier 1 and tier 2 already
  resolved should NOT be re-sent here.

Requires OPENAI_API_KEY (chat completions, real per-call cost here,
unlike tiers 1 and 2).
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# GENERAL check, not a hardcoded word list. A fixed vocabulary (gel,
# commercial, native, ...) only catches failure modes we've already
# seen, at full corpus scale (12,000 papers) a new distinguishing word
# we never anticipated (roasted, fermented, extruded, encapsulated...)
# would sail straight past it. Instead: compute which words are shared
# by EVERY member of a group versus which words appear in only SOME
# members. Any asymmetric word is a candidate real distinction, purely
# structural, no vocabulary required, so it generalizes to any future
# wording difference, not just the ones we've caught so far.
_STOPWORDS = {"from", "of", "the", "a", "an", "and", "or", "in", "on", "at", "to", "for", "with"}


def tokenize(entity_name):
    words = re.findall(r"[a-z0-9\-]+", str(entity_name).lower())
    return {w for w in words if w not in _STOPWORDS}


def get_asymmetric_tokens(members):
    """
    Words that appear in some members but not all, the general
    structural signal that something distinguishing might differ
    between them, without assuming which words matter in advance.
    """
    token_sets = [tokenize(m) for m in members]
    shared_core = set.intersection(*token_sets) if token_sets else set()
    all_tokens = set.union(*token_sets) if token_sets else set()
    return sorted(all_tokens - shared_core)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "expanded_triples.xlsx"  # output of consolidate_resolutions.py, NOT the raw triples file
SOURCE_COL = "expanded_source"  # abbreviation-resolved entity text, not the raw "source" column
TARGET_COL = "expanded_target"
DOI_COL = "doi"
SENTENCE_COL = "corresponding_sentence"

# Input: candidate groups needing adjudication, pulled from BOTH of
# tier 2's "couldn't resolve automatically" sheets, not just one.
# skipped_state_conflict and skipped_low_similarity are both genuine
# "needs a closer look" cases, so both get processed here in one pass.
CANDIDATES_PATH = "tier2_llm_code_extraction_review.xlsx"  # output of tier2_llm_code_extraction.py
CANDIDATES_SHEETS = ["skipped_state_conflict", "skipped_low_similarity"]
GROUP_COL = "code"     # column identifying which rows belong to the same group
DOI_KEY_COL = "doi"

LLM_MODEL = "gpt-4o-mini"
OUTPUT_XLSX = "tier3_llm_adjudication_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# LOAD TRIPLES (for sentence lookup) AND CANDIDATE GROUPS FROM BOTH SHEETS
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)

rows = []
for sheet in CANDIDATES_SHEETS:
    candidates = pd.read_excel(CANDIDATES_PATH, sheet_name=sheet)
    for _, r in candidates.iterrows():
        for m in str(r["members"]).split("; "):
            rows.append({"doi": r[DOI_KEY_COL], "group": r[GROUP_COL],
                         "entity": m.strip(), "source_sheet": sheet})
candidate_long = pd.DataFrame(rows)

print(f"Loaded {len(candidate_long)} entities across "
      f"{candidate_long[['doi','group']].drop_duplicates().shape[0]} groups needing adjudication "
      f"(from {len(CANDIDATES_SHEETS)} tier 2 sheets: {', '.join(CANDIDATES_SHEETS)})")


def get_sentences(entity, doi):
    mask = ((df[SOURCE_COL].astype(str).str.strip() == entity) |
            (df[TARGET_COL].astype(str).str.strip() == entity)) & (df[DOI_COL] == doi)
    return df.loc[mask, SENTENCE_COL].dropna().astype(str).unique().tolist()


# ---------------------------------------------------------------
# LLM ADJUDICATION
# ---------------------------------------------------------------
ADJUDICATION_PROMPT = """You are resolving entity names in a plant protein functional
properties knowledge graph. These entity names were flagged as candidate variants of
the same underlying entity, but could not be auto-resolved, so they need judgment.

All entities below come from the SAME paper (DOI: {doi}).

Candidate entities:
{members}
{asymmetric_note}
Sentence context where each entity appears:
{sentences}

Decide:
1. Are these genuinely the SAME real-world sample/entity referred to differently, or
   are they DISTINCT samples/entities (e.g. different processing states like raw vs
   gelled, different genotypes) that happen to share similar wording? Use the sentence
   context, not just the wording, to decide. Be willing to say they are distinct.
   If a word-difference list is shown above, you MUST explicitly address each one:
   state whether it reflects a real distinction or is inconsequential phrasing, do not
   let it get lost if most entities in the group agree with each other, a minority
   variant with a genuine distinction is still a genuine distinction.
2. If they are the same entity, propose ONE canonical name.
3. One-sentence justification citing what in the sentences supports the decision, and
   explicitly addressing any word-difference listed above.

Respond ONLY with JSON, no markdown fences:
{{"is_same_entity": true/false, "canonical_name": "..." or null, "justification": "..."}}
"""


def adjudicate(doi, members, sentences_by_entity, asymmetric_tokens):
    sentence_block = "\n".join(
        f"- {m}: " + (" | ".join(sentences_by_entity.get(m, [])[:2]) or "(no sentence found)")
        for m in members
    )
    if asymmetric_tokens:
        asymmetric_note = (
            "\nWord-difference check: these words appear in SOME but not ALL of the "
            f"entities above: {', '.join(asymmetric_tokens)}. You must directly address "
            "whether each one signals a real distinction before deciding.\n"
        )
    else:
        asymmetric_note = ""
    prompt = ADJUDICATION_PROMPT.format(
        doi=doi,
        members="\n".join(f"- {m}" for m in members),
        asymmetric_note=asymmetric_note,
        sentences=sentence_block,
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    time.sleep(0.2)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"is_same_entity": None, "canonical_name": None, "justification": f"PARSE_FAILED: {raw[:200]}"}


results = []
for (doi, group), members_df in candidate_long.groupby(["doi", "group"]):
    members = members_df["entity"].tolist()
    source_sheet = members_df["source_sheet"].iloc[0]
    sentences_by_entity = {m: get_sentences(m, doi) for m in members}
    asymmetric_tokens = get_asymmetric_tokens(members)
    verdict = adjudicate(doi, members, sentences_by_entity, asymmetric_tokens)

    is_same = verdict.get("is_same_entity")
    canonical_name = verdict.get("canonical_name")
    justification = verdict.get("justification")

    # Not a hard override, since we can't assume every asymmetric word
    # is disqualifying (plurals, ID-completeness differences are often
    # harmless, and a fixed rule for which words are "safe" is exactly
    # the kind of narrow assumption we're trying to avoid). Instead:
    # flag the risky combination, claimed sameness despite an unresolved
    # wording difference, for mandatory human review rather than
    # trusting it silently. This generalizes to any future word, since
    # it's asking "did wording differ AND did the model claim sameness
    # anyway", not "does this specific word appear".
    needs_review = bool(is_same is True and asymmetric_tokens)

    results.append({
        "doi": doi,
        "group": group,
        "source_sheet": source_sheet,
        "members": "; ".join(members),
        "asymmetric_tokens": "; ".join(asymmetric_tokens),
        "is_same_entity": is_same,
        "proposed_canonical_name": canonical_name,
        "justification": justification,
        "needs_manual_review": needs_review,
    })

review_df = pd.DataFrame(results)
n_same = (review_df["is_same_entity"] == True).sum()  # noqa: E712
n_distinct = (review_df["is_same_entity"] == False).sum()  # noqa: E712
print(f"\nLLM judged {n_same} groups as the same entity, {n_distinct} as genuinely distinct")

readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    f"This file adjudicates candidate groups pulled from BOTH of tier 2's unresolved sheets in",
    f"{CANDIDATES_PATH}: {' and '.join(CANDIDATES_SHEETS)}. Both are genuine",
    "\"needs a closer look\" cases, so both get processed here in one pass, not just one sheet.",
    "",
    "- doi / group: which paper and which shared code/cluster this group of entities came from.",
    "",
    "- source_sheet: which tier 2 sheet this group came from originally, skipped_state_conflict",
    "  (members disagreed on a word like \"gel\") or skipped_low_similarity (code matched but text",
    "  similarity was too low to trust). Kept for traceability back to why tier 2 flagged it.",
    "",
    "- members: the entity names in this group, exactly as they appeared going in, semicolon-separated.",
    "",
    "- asymmetric_tokens: words that appear in SOME but not ALL members of this group, computed",
    "  automatically (no fixed word list), a general signal that something might genuinely distinguish",
    "  them (a processing state, a material form) rather than being pure phrasing variation. This is",
    "  what gets explicitly pointed out to the LLM before it decides, so a minority variant doesn't get",
    "  averaged away by the majority the way it did before this check existed.",
    "",
    "- is_same_entity: the LLM's judgment, using the actual sentence(s) each entity appeared in as evidence,",
    "  not just the wording. True = genuinely the same real-world sample/object described two ways.",
    "  False = genuinely distinct (e.g. different processing states or different genotypes that happen to",
    "  share similar wording). The LLM was explicitly told it's allowed to say False, this is not a forced",
    "  merge tool.",
    "",
    "- proposed_canonical_name: only meaningful when is_same_entity = True, the suggested single name to",
    "  merge the group's members into. Blank/irrelevant when is_same_entity = False.",
    "",
    "- justification: one sentence citing what in the actual source sentences supports the decision.",
    "  Spot-check a sample of both True and False verdicts against this before trusting the file, this",
    "  is the last and most expensive tier, but it can still be wrong.",
    "",
    "- needs_manual_review: True when the LLM said \"same entity\" DESPITE an asymmetric word being present.",
    "  This is the riskiest combination and is not auto-trusted either way, review these rows by hand",
    "  before applying the merge, regardless of what the justification says.",
]
readme_df = pd.DataFrame({"": readme_rows})

with pd.ExcelWriter(OUTPUT_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    review_df.to_excel(writer, sheet_name="adjudication_results", index=False)

print(f"Saved to {OUTPUT_XLSX}")
print("Spot-check a sample of both same-entity and distinct-entity verdicts before trusting them.")

Loaded 67 entities across 21 groups needing adjudication (from 2 tier 2 sheets: skipped_state_conflict, skipped_low_similarity)

LLM judged 0 groups as the same entity, 21 as genuinely distinct
Saved to tier3_llm_adjudication_review.xlsx
Spot-check a sample of both same-entity and distinct-entity verdicts before trusting them.
